In [95]:
import xarray as xr
import pandas as pd
from tqdm import tqdm
from glob import glob
from braceexpand import braceexpand
import numpy as np
import pickle

In [85]:
sim = 'ERA5'

if sim in ['ERA5', 'UBB']:
    endyear = 2023
else:
    endyear = 2014

In [86]:
def braced_glob(path):
    l = []
    for x in braceexpand(path):
        l.extend(glob(x))          
    return l

In [87]:
def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    Possible and slightly more precise with geopy or pyproj through xr.apply_ufunc but longer
    """
    # convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    # haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6367 * c
    return km

In [88]:
if sim == 'ERA5':
    df = pd.read_csv('/home/vdemeyer/projects/rrg-gachon/vdemeyer/TRACKING/KATJA/OUTPUTS/ERA5_CORDEX_NA_psl_smooth_400km_12h_990hPa.txt',
            sep=r'\s+',header=15, engine='python', names=['storm','i','j','point', 'date','lat','lon','pressure'])
else:
    df = pd.read_csv(f'/home/vdemeyer/projects/rrg-gachon/vdemeyer/TRACKING/KATJA/OUTPUTS/{sim}_psl_smooth_400km_12h_990hPa.txt',
                    sep=r'\s+',header=15, engine='python', names=['storm','i','j','point', 'date','lat','lon','pressure'])
df['date'] = pd.to_datetime(df['date'])

In [89]:
with open(f'/home/vdemeyer/projects/rrg-gachon/vdemeyer/TRACKING/KATJA/OUTPUTS/EETC/EETC_cum_{sim}_Quebec_1979-{endyear}_compound_8hrs_quantile_SSI.pkl', 'rb') as pickle_file:
    EETC_dict = pickle.load(pickle_file)

In [90]:
if sim == 'ERA5':
    percentile_precip = xr.open_dataset(f'/home/vdemeyer/projects/rrg-gachon/vdemeyer/{sim}/PR_Percentile/tp_{sim.lower()}_percentile.nc')
    percentile_precip = percentile_precip.rename({'tp': 'pr'})
else:
    percentile_precip = xr.open_dataset(f'/home/vdemeyer/projects/rrg-gachon/vdemeyer/{sim}/PR_Percentile/pr_{sim.lower()}_percentile.nc')
    percentile_precip['pr'] = percentile_precip.pr * 3600.
percentile_precip = percentile_precip.sel(quantile=[0.999])

percentile_wind = xr.open_dataset(f'/home/vdemeyer/projects/rrg-gachon/vdemeyer/{sim}/WIND_Percentile/surf_wind_{sim.lower()}_percentile.nc')
percentile_wind = percentile_wind.sel(quantile=[0.999])
percentile_wind['surf_wind'] = percentile_wind.surf_wind * 3.6

In [91]:
west_lon = -80
east_lon = -66
north_lat = 55
south_lat = 44

if sim == 'ERA5':
    percentile_precip = percentile_precip.sel({'longitude': slice(west_lon, east_lon), 'latitude': slice(north_lat, south_lat)})
    percentile_wind = percentile_wind.sel({'longitude': slice(west_lon, east_lon), 'latitude': slice(north_lat, south_lat)})

else:
    mask = (
        (percentile_precip.lon >= west_lon) &
        (percentile_precip.lon <= east_lon) &
        (percentile_precip.lat >= south_lat) &
        (percentile_precip.lat <= north_lat)
    )

    percentile_precip = percentile_precip.where(mask, drop=True)
    percentile_wind = percentile_wind.where(mask, drop=True)

In [92]:
coord_lon = 'longitude' if sim == 'ERA5' else 'lon'
coord_lat = 'latitude' if sim == 'ERA5' else 'lat'

all_storms_df = pd.DataFrame()

# for id_storm in tqdm(EETC_dict):
if sim == 'UBB': id_storm = 6744 #halloween UBB
if sim == 'ERA5': id_storm = 6751 #halloween ERA5

if EETC_dict[id_storm]['cum_precip'].sel(quantile=0.999).values > 0 and EETC_dict[id_storm]['cum_wind'].sel(quantile=0.999).values > 0:

    iEETC = df.groupby('storm').get_group(id_storm)

    month_years = iEETC['date'].dt.to_period('M').unique()

    if sim == 'ERA5':
        filenames_precip = '/home/vdemeyer/projects/rrg-gachon/vdemeyer/ERA5/PR/era5_tp_CORDEX_NA_1979-2023.zarr'
        ds_precip = xr.open_mfdataset(filenames_precip)
        ds_precip = ds_precip.rename({'tp': 'pr'})
        ds_precip = ds_precip.sel(time=ds_precip['time'].dt.strftime('%Y-%m').isin(month_years.astype(str)))

        filenames_wind = '/home/vdemeyer/projects/rrg-gachon/vdemeyer/ERA5/WIND/Magnitude/era5_wind10_CORDEX_NA_1979-2023.zarr'
        ds_wind = xr.open_mfdataset(filenames_wind)
        ds_wind = ds_wind.sel(time=ds_wind['time'].dt.strftime('%Y-%m').isin(month_years.astype(str)))

    else:
        filename_precip = []
        filename_wind = []
        for period in month_years:
            year = period.year
            month = period.month
            filename_precip.extend(braced_glob(f'/home/vdemeyer/projects/rrg-gachon/vdemeyer/{sim}/PR/pr_{sim.lower()}_{year}{month:02d}_se.nc'))
            filename_wind.extend(braced_glob(f'/home/vdemeyer/projects/rrg-gachon/vdemeyer/{sim}/WIND/wind10_{sim.lower()}_{year}{month:02d}_se.nc'))
        ds_precip = xr.open_mfdataset(filename_precip)
        ds_precip['time'] = ds_precip['time'].dt.floor('h')
        ds_precip['pr'] = ds_precip.pr * 3600.
        
        ds_wind = xr.open_mfdataset(filename_wind)
        ds_wind['time'] = ds_wind['time'].dt.floor('h')
    ds_wind['surf_wind'] = ds_wind.surf_wind * 3.6

    mask_diff_precip = ds_precip - percentile_precip
    mask_diff_precip = mask_diff_precip.where(mask_diff_precip >= 0, np.nan)
    mask_diff_precip = mask_diff_precip.compute()
    mask_diff_wind = ds_wind - percentile_wind
    mask_diff_wind = mask_diff_wind.where(mask_diff_wind >= 0, np.nan)
    mask_diff_wind = mask_diff_wind.compute()

    itime_precip_first = None
    itime_wind_first = None
    itime_precip_last = None
    itime_wind_last = None

    for itrack, row in iEETC.iterrows():        
        ilon, ilat, itime = row['lon'], row['lat'], row['date']

        if itime_precip_first is None:
            mask_itime_precip = mask_diff_precip.sel(time=itime).where(mask_diff_precip.sel(time=itime).map(lambda x: haversine(ilon, ilat, getattr(x, coord_lon), getattr(x, coord_lat)) <= 1000), drop=True).pr
            if mask_itime_precip.notnull().any():
                itime_precip_first = itime

        if itime_wind_first is None:
            mask_itime_wind = mask_diff_wind.sel(time=itime).where(mask_diff_wind.sel(time=itime).map(lambda x: haversine(ilon, ilat, getattr(x, coord_lon), getattr(x, coord_lat)) <= 1000), drop=True).surf_wind
            if mask_itime_wind.notnull().any():
                itime_wind_first = itime

        if itime_precip_first is not None and itime_wind_first is not None:
            break

    for itrack, row in iEETC.iloc[::-1].iterrows():        
        ilon, ilat, itime = row['lon'], row['lat'], row['date']

        if itime_precip_last is None:
            mask_itime_precip = mask_diff_precip.sel(time=itime).where(mask_diff_precip.sel(time=itime).map(lambda x: haversine(ilon, ilat, getattr(x, coord_lon), getattr(x, coord_lat)) <= 1000), drop=True).pr
            if mask_itime_precip.notnull().any():
                itime_precip_last = itime

        if itime_wind_last is None:
            mask_itime_wind = mask_diff_wind.sel(time=itime).where(mask_diff_wind.sel(time=itime).map(lambda x: haversine(ilon, ilat, getattr(x, coord_lon), getattr(x, coord_lat)) <= 1000), drop=True).surf_wind
            if mask_itime_wind.notnull().any():
                itime_wind_last = itime

        if itime_precip_last is not None and itime_wind_last is not None:
            break

    if itime_precip_first is not None and itime_wind_first is not None and itime_precip_last is not None and itime_wind_last is not None:
        new_df = pd.DataFrame({
            'id_storm': [id_storm],
            'ETC_PRcum': [EETC_dict[id_storm]['cum_precip'].sel(quantile=0.999).values],
            'ETC_PRint': [EETC_dict[id_storm]['cum_avg_precip'].sel(quantile=0.999).values],
            'ETC_WScum': [EETC_dict[id_storm]['cum_wind'].sel(quantile=0.999).values],
            'ETC_WSint': [EETC_dict[id_storm]['cum_avg_wind'].sel(quantile=0.999).values],
            'compound_PR_WS': [EETC_dict[id_storm]['compound_8hrs_quantile']['99.9']],
            'first_date_with_PRext': [itime_precip_first.strftime('%Y-%m-%d %H:%M:%S')],
            'first_date_with_WSext': [itime_wind_first.strftime('%Y-%m-%d %H:%M:%S')],
            'last_date_with_PRext': [itime_precip_last.strftime('%Y-%m-%d %H:%M:%S')],
            'last_date_with_WSext': [itime_wind_last.strftime('%Y-%m-%d %H:%M:%S')]
        })

        all_storms_df = pd.concat([all_storms_df, new_df], ignore_index=True)

all_storms_df
# all_storms_df.to_csv(f'/home/vdemeyer/projects/rrg-gachon/vdemeyer/TRACKING/KATJA/OUTPUTS/EETC_{sim}_Quebec_1979-{endyear}_per999_for_Alejandro.txt', index=False)

itrack
itrack
itrack
itrack
itrack
itrack
itrack
itrack
itrack
itrack
itrack
itrack
itrack
itrack
itrack
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2
itrack2


,id_storm,ETC_PRcum,ETC_PRint,ETC_WScum,ETC_WSint,compound_PR_WS,first_date_with_PRext,first_date_with_WSext,last_date_with_PRext,last_date_with_WSext
0,6751,1.3789944745296265,0.34527172130738787,7.0581627,1.1588981957121198,True,2019-10-31 18:00:00,2019-11-01 02:00:00,2019-11-01 22:00:00,2019-11-02 01:00:00
